# Research source: Grouped learning, calibration and affine export

**Route:** archival methodology, not the default frozen-deployment quick start.  
**Original uploaded notebook:** `02_LP35_Train_and_Export(2).ipynb`.  
**Purpose:** Keeps whole parent groups together; constructs out-of-fold scores and one calibrated affine rule per selected classifier; exports the frozen GAMS interface.

## Before executing
The public optimizer uses the supplied numerical includes and frozen coefficients. It **does not need this notebook to be rerun**. Start with `notebooks/00_Quick_Start.ipynb` and `GAMS_LaPalma/START_LAPALMA.gms` instead.

Full execution of this research notebook needs: Branch labels and protocol; matching learning receipts; fitted artifacts for exact replay; 56 original BC/RESERVE days for the original export guard. These archival dependencies are not all in the lightweight public package. The exact original case/protocol bytes have not been fabricated. Original source-hash checks are intentionally strict and may reject comment-sanitized public copies; archival replay uses the exact private originals and the author's matching data archive.

## Governing model and units
UC dispatch and reserve are in MW; surviving inertia is in MW s; gain is in MW/Hz. The primary frequency criterion is a 46.5-Hz minimum at 50 Hz. The historical notebook branch remains 47 Hz. The SODE model uses one frequency state and two states per surviving governor; its response is limited above by individual headroom. The feature vector order is `loss_mw, inertia_mws, gain_mw_hz, reserve_mw`. A calibrated score becomes an affine GAMS inequality after inverse standardization and a probability-logit shift.

## Algorithm, interpretation and verification
Read the original Markdown and the stage annotations below in order. The detailed derivations and symbol-to-source mapping are in `docs/Mathematical_Formulation/Technical_Manual.pdf`. A grouping or calibration diagnostic is not a physical risk guarantee. Do not transfer a previous day's state through rounded Excel output; use native GDX.

**Curation status:** notebook structure, imports, Python syntax and exact preservation of all original code-cell text are checked. Full production execution is **NOT RUN**. Source-only functions may be exercised by the small verification notebooks using explicitly labelled fixtures; that does not validate the unavailable full experiment.

Original outputs and execution counts were removed from this public copy to avoid stale results and private workstation paths. The separately supplied private original backup preserves the original bytes.


# 02 — Fit, calibrate and export the new 3.5-Hz rules

## Numerical-calibration repair: resume, do not restart the experiment
The saved failure was an `ABNORMAL` optimization termination while fitting the
**final out-of-fold sigmoid for random-selection SVM C=0.1**. The four-line
training cell below was not the defective code.

This replacement changes **only `sigmoid_from_oof()`**: it keeps the same smoothed
Platt objective, data, groups and monotone sign restriction, with checked stable
fallbacks when the original numerical optimizer cannot confirm convergence.
Completed candidate `.joblib` checkpoints remain unchanged and are reused.
An interrupted candidate must be retried because the old code did not checkpoint
its inner models before the final calibration. Remaining candidates then continue.

**Use your existing Anaconda environment and leave the `Data_LaPalma` folder in place.**
Restart Kernel → Run All. Do not rerun Notebook 01, GAMS, or the 3-Hz notebooks.
The training cell should first print `Reusing candidate checkpoint:` for completed
models, then resume the interrupted/remaining candidates.

This repair does not make the rare-event sample statistically stronger. It does
not relax the 3.5-Hz physics, change labels or replace a noninformative rule with
a constant-safe one. Do not delete the export or group-separation checks.


## Actual validation design
Combine only the old development and calibration parents. Inner three-fold **parent** cross-fitting supplies out-of-fold scores for an increasing, smoothed sigmoid, followed by one final estimator. Outer three-fold evaluation repeated twice evaluates this complete fitted-and-calibrated procedure. The cost/random half-filter applies only to fold fitting, never to its scoring/evaluation parents.

A noninformative fold is retained as `FAILED_RETAINED`, not replaced by a constant-safe rule. A final deployment candidate must have a valid increasing calibrator and pass coefficient equivalence. Test FNR denominators remain small and are disclosed. This is a documented change from the earlier single reserved-calibration subset.

### Source code stage 3

**Algorithm objects:** `find_project`, `sha_file`, `hash_object`, `atomic_json`, `atomic_csv`, `write_excel`, `excel_values`, `number`, `read_matrix`, `read_pairs`, `assert_sources`, `event_inputs`, `event_key`, `integrate_event`, `evaluate_event`, `cache_open`, `cache_save`.

These definitions implement the original routines; inspect their input validation and return fields before calling them. Defining a function does not itself verify its numerical behavior.

**Verification:** original code-cell text is unchanged and syntax-checked. Printed values and figures become new evidence only after their required inputs have been supplied and execution actually succeeds. Preserve caches/receipts and distinguish a failed calculation from an unsafe physical label.

In [ ]:
# Shared definitions are embedded verbatim in the notebooks; no custom .py is needed.
from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED
from xml.etree import ElementTree as ET
import csv, hashlib, json, math, os, re, shutil, sys, time, tempfile
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from scipy.optimize import minimize_scalar
from scipy.special import expit


def find_project():
    here = Path.cwd().resolve()
    for p in (here, here.parent):
        if (p/'Data_LaPalma/LP_Case.json').is_file() and (p/'GAMS_LaPalma').is_dir():
            return p
    raise FileNotFoundError('Open this notebook in LaPalma_SEGAN/Notebooks_LaPalma, not the IEEE project.')

PROJECT = find_project()
NB = PROJECT/'Notebooks_LaPalma'
DATA = PROJECT/'Data_LaPalma'
GAMS = PROJECT/'GAMS_LaPalma'
EVID = PROJECT/'Evidence'; EVID.mkdir(exist_ok=True)
ORIGINAL_CASE = json.loads((DATA/'LP_Case.json').read_text(encoding='utf-8'))
CASE = json.loads(json.dumps(ORIGINAL_CASE))
CASE['nadir_min_hz'] = 46.5  # New branch only: do not edit LP_Case.json on disk.
IDS = CASE['ids']; NGEN = len(IDS)
if IDS != [f'i{j}' for j in range(1, 12)]:
    raise ValueError('Expected the original ordered eleven-unit La Palma fleet.')
PMAX = np.array(CASE['pmax']); PMIN = np.array(CASE['pmin'])
HS = np.array(CASE['inertia_s']); MBASE = np.array(CASE['mbase'])
KG = np.array(CASE['gain_pu']); A1 = np.array(CASE['a1'])
A2 = np.array(CASE['a2']); B1 = np.array(CASE['b1'])
F0 = float(CASE['f0_hz']); FMIN = float(CASE['nadir_min_hz'])
if not (np.isclose(F0,50.) and np.isclose(F0-FMIN,3.5)):
    raise ValueError('This SEGAN La Palma design requires 50 Hz and a 3.5-Hz drop (46.5-Hz threshold).')
EUNIT = HS*MBASE
GAIN = KG*MBASE/F0  # MW/Hz; source gain K*Mbase = F0*GAIN.
for vector in (PMAX, PMIN, HS, MBASE, KG, A1, A2, B1):
    if vector.shape != (11,) or not np.isfinite(vector).all():
        raise ValueError('Every unit-parameter vector must contain exactly eleven finite entries.')
CORE_FEATURES = ['loss_mw', 'inertia_mws', 'gain_mw_hz', 'reserve_mw']
NUMERIC = dict(method='LSODA', rtol=1e-8, atol=1e-10, max_step=.20,
               output_step=.02, duration=60., extension=120.,
               stop_hz=40., settle_range_hz=.005, settle_slope_hz_s=1e-4,
               nadir_refine_tol_hz=1e-3, boundary_hz=1e-4)
DYNAMIC_VERSION = 'LP_UNIT_ID_UPPER_CAP_HZ_SOURCE_DAMP_v3'
MODEL_SPEC = {k:CASE[k] for k in ['ids','pmax','inertia_s','mbase','gain_pu','a1','a2','b1',
                                      'f0_hz','damping_fraction_per_hz','nadir_min_hz',
                                      'rocof_max_hz_s','qss_drop_hz']}
MODEL_SPEC.update(version=DYNAMIC_VERSION, limiter='minimum(internal_response, unit_headroom)',
                  numeric=NUMERIC, units='Hz / MW / MW*s / MW-per-Hz')

def sha_file(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(1024*1024),b''): h.update(block)
    return h.hexdigest()

def hash_object(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()

MODEL_ID = hash_object(MODEL_SPEC)[:16]
WORK = DATA/'Threshold35'/MODEL_ID; WORK.mkdir(parents=True,exist_ok=True)
LEARN = WORK/'Learning'; LEARN.mkdir(exist_ok=True)
OLD_SPEC = json.loads(json.dumps(MODEL_SPEC)); OLD_SPEC['nadir_min_hz']=float(ORIGINAL_CASE['nadir_min_hz'])
OLD_MODEL_ID=hashlib.sha256(json.dumps(OLD_SPEC,sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()[:16]
OLD_WORK=DATA/'Corrected'/OLD_MODEL_ID
BRANCH_VERSION='LP35_OOF_AFFINE_20260923_v1'
print('3.5-Hz branch:',MODEL_ID,'| reusing source library:',OLD_MODEL_ID,flush=True)

def atomic_json(path, value):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=lambda x:x.item() if isinstance(x,np.generic) else str(x)),encoding='utf-8')
    os.replace(tmp,path)

def atomic_csv(frame,path):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp');frame.to_csv(tmp,index=False);os.replace(tmp,path)

def write_excel(path, sheets):
    """CSV/checkpoints are authoritative. Write/validate a NEW workbook, then replace.
    A Windows-locked old workbook is not deleted and never costs another simulation.
    All sheet names are keyword arguments (pandas 3-compatible).
    """
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.stem+'_writing_'+str(time.time_ns())+'.xlsx')
    try:
        with pd.ExcelWriter(tmp,engine='openpyxl') as writer:
            for name,frame in sheets.items():
                if not isinstance(frame,pd.DataFrame):frame=pd.DataFrame(frame)
                frame.to_excel(writer,sheet_name=name[:31],index=False)
                ws=writer.sheets[name[:31]];ws.freeze_panes='A2'
                ws.auto_filter.ref=ws.dimensions
                from copy import copy as copy_style
                for c in ws[1]:
                    font=copy_style(c.font); font.bold=True; c.font=font
                for column_index in range(1,ws.max_column+1):
                    col=[ws.cell(row=row_index,column=column_index) for row_index in range(1,min(100,ws.max_row)+1)]
                    samples=[len(str(c.value)) for c in col if c.value is not None]
                    ws.column_dimensions[col[0].column_letter].width=min(38,max(12,max(samples,default=12)+2))
        with ZipFile(tmp) as z:
            if z.testzip() is not None or 'xl/workbook.xml' not in z.namelist():raise IOError('Invalid Excel export')
        try: os.replace(tmp,path)
        except PermissionError:
            path=path.with_name(path.stem+'_RECOVERED_'+str(time.time_ns())+'.xlsx');os.replace(tmp,path)
            print('The prior workbook is locked; saved a fresh workbook:',path)
    except Exception:
        print('Excel export failed. Saved CSVs/caches are retained; rerun only the output cell.')
        raise
    return path

# Reader uses stored cell values, including GAMS EPS, with no Excel automation.
def excel_values(path, sheet):
    ns={'x':'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
    rid='{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id'
    with ZipFile(path) as z:
        strings=[]
        if 'xl/sharedStrings.xml' in z.namelist():
            strings=[''.join(x.itertext()) for x in ET.fromstring(z.read('xl/sharedStrings.xml')).findall('x:si',ns)]
        rel={x.attrib['Id']:x.attrib['Target'] for x in ET.fromstring(z.read('xl/_rels/workbook.xml.rels'))}
        sheets=ET.fromstring(z.read('xl/workbook.xml')).find('x:sheets',ns)
        target=next((rel[s.attrib[rid]] for s in sheets if s.attrib['name']==sheet),None)
        if target is None:raise KeyError(f'{Path(path).name}: missing sheet {sheet}')
        target=target.lstrip('/') if target.startswith('/') else 'xl/'+target
        rows=[]
        for row in ET.fromstring(z.read(target)).findall('x:sheetData/x:row',ns):
            d={}
            for c in row:
                col=0
                for a in re.match(r'[A-Z]+',c.attrib['r']).group():col=col*26+ord(a)-64
                v=c.find('x:v',ns);typ=c.attrib.get('t')
                if typ=='s':value=strings[int(v.text)] if v is not None else None
                elif typ=='inlineStr':value=''.join(c.find('x:is',ns).itertext())
                elif v is None:value=None
                elif typ in ('str','e'):value=v.text
                else:
                    try:value=float(v.text)
                    except (ValueError,TypeError):value=v.text
                d[col-1]=value
            rows.append(d)
        width=max((max(r,default=-1)+1 for r in rows),default=0)
        return [[r.get(j) for j in range(width)] for r in rows]

def number(value):
    if value is None or value=='':return 0.
    if isinstance(value,str) and value.strip().lower() in ('eps','epsilon'):return 0.
    out=float(value)
    if not math.isfinite(out):raise ValueError(f'Nonfinite native number: {value}')
    return out

def read_matrix(path,sheet,row_ids,col_ids):
    data=excel_values(path,sheet);result=np.zeros((len(row_ids),len(col_ids)))
    headers=next((r for r in data if sum(str(x) in col_ids for x in r)>=1),None)
    if headers is None:raise ValueError(f'{sheet}: no expected column IDs')
    positions={str(x):j for j,x in enumerate(headers) if str(x) in col_ids}
    if set(positions)!=set(col_ids):raise ValueError(f'{sheet}: incomplete column identifiers')
    mapped={str(r[0]):r for r in data if r and str(r[0]) in row_ids}
    if set(mapped)!=set(row_ids):raise ValueError(f'{sheet}: incomplete row identifiers')
    for i,key in enumerate(row_ids):
        for j,col in enumerate(col_ids):result[i,j]=number(mapped[key][positions[col]])
    return result

def read_pairs(path,sheet,expected=None):
    result={}
    for row in excel_values(path,sheet):
        if len(row)>=2 and row[0] is not None and (expected is None or str(row[0]) in expected):
            try:result[str(row[0])]=number(row[1])
            except (ValueError,TypeError):continue
    return result

def assert_sources():
    for relative,expected in CASE['source_hashes'].items():
        actual=sha_file(PROJECT/relative)
        if actual!=expected:raise ValueError(f'Original source changed: {relative}; do not reuse this case identity.')




def event_inputs(power, demand_mw, outage_index, commitment=None):
    power=np.asarray(power,float).copy();demand_mw=float(demand_mw);j=int(outage_index)
    if power.shape!=(11,) or not np.isfinite(power).all() or demand_mw<0:raise ValueError('Invalid event input')
    on=(power>1e-8) if commitment is None else np.rint(np.asarray(commitment,float)).astype(bool)
    if on.shape!=(11,) or not on[j] or power[j]<=1e-8:raise ValueError('Outage must remove an online producing unit')
    if np.any(power<-1e-6) or np.any(power>PMAX*on+1e-6):raise ValueError('Invalid pre-event generator outputs')
    survive=on.copy();survive[j]=False
    idx=np.flatnonzero(survive);E=float(EUNIT[survive].sum());rho=np.maximum(PMAX-power,0)[idx]
    return dict(power=power,online=on,survive=survive,index=idx,E=E,rho=rho,
        gain=GAIN[idx],a1=A1[idx],a2=A2[idx],b1=B1[idx],loss=float(power[j]),
        damping=demand_mw*CASE['damping_fraction_per_hz'],demand=demand_mw,outage=j)


def event_key(power,demand_mw,outage_index,commitment=None):
    on=(np.asarray(power)>1e-8) if commitment is None else np.rint(commitment).astype(bool)
    identity=dict(model_id=MODEL_ID,power=np.round(power,9).tolist(),demand=round(float(demand_mw),9),
                  outage=IDS[int(outage_index)],commitment=on.astype(int).tolist())
    return hash_object(identity)


def integrate_event(power,demand_mw,outage_index,commitment=None,refined=False,return_trace=False):
    v=event_inputs(power,demand_mw,outage_index,commitment)
    if v['E']<=0:raise ValueError('No surviving inertia; no fictitious epsilon inertia is inserted')
    n=len(v['index']);rtol=NUMERIC['rtol']*(.1 if refined else 1.)
    atol=NUMERIC['atol']*(.1 if refined else 1.)
    step=NUMERIC['output_step']*(.5 if refined else 1.)
    mx=NUMERIC['max_step']*(.5 if refined else 1.)
    def response(y):return np.minimum(v['gain']*(y[1:1+n]+v['b1']*y[1+n:]),v['rho'])
    def rhs(t,y):
        df=y[0];q=y[1:1+n];dq=y[1+n:]
        dfdt=F0/(2*v['E'])*(response(y).sum()-v['loss']-v['damping']*df)
        ddq=(-q-v['a1']*dq-df)/v['a2']
        return np.concatenate(([dfdt],dq,ddq))
    def stop(t,y):return F0+y[0]-NUMERIC['stop_hz']
    stop.terminal=True;stop.direction=-1
    y0=np.zeros(1+2*n);duration=NUMERIC['duration'];pieces=[];start=0.;state=y0
    while True:
        sol=solve_ivp(rhs,(start,duration),state,method=NUMERIC['method'],rtol=rtol,atol=atol,
                      max_step=mx,dense_output=True,events=stop)
        if not sol.success or not np.isfinite(sol.y).all():raise RuntimeError(sol.message)
        end=float(sol.t[-1]);grid=np.linspace(start,end,max(2,int(np.ceil((end-start)/step))+1))
        Y=sol.sol(grid);frequency=F0+Y[0];pieces.append((grid,Y,sol))
        censored=bool(sol.t_events[0].size)
        tail=grid>=max(start,end-5.)
        slope=float(rhs(end,Y[:,-1])[0]);range_=float(np.ptp(frequency[tail]))
        settled=(not censored and end>=10 and range_<=NUMERIC['settle_range_hz'] and abs(slope)<=NUMERIC['settle_slope_hz_s'])
        if censored or settled or duration>=NUMERIC['extension']:break
        start=end;state=Y[:,-1];duration=NUMERIC['extension']
    times=np.concatenate([g if k==0 else g[1:] for k,(g,_,__) in enumerate(pieces)])
    states=np.concatenate([y if k==0 else y[:,1:] for k,(_,y,__) in enumerate(pieces)],axis=1)
    freq=F0+states[0];nadir=float(freq.min());tn=float(times[freq.argmin()])
    # Refine each sampled local minimum using the dense interpolation of its own interval.
    for grid,Y,sol in pieces:
        f=F0+Y[0]
        for k in np.flatnonzero((f[1:-1]<f[:-2])&(f[1:-1]<=f[2:]))+1:
            opt=minimize_scalar(lambda t:float(F0+sol.sol(t)[0]),bounds=(grid[k-1],grid[k+1]),method='bounded',options={'xatol':1e-9})
            if opt.fun<nadir:nadir=float(opt.fun);tn=float(opt.x)
    rtotal=np.array([response(y).sum() for y in states.T])
    record=dict(minimum_observed_hz=nadir,time_min_s=tn,endpoint_hz=float(freq[-1]),
        initial_rocof_hz_s=float(rhs(0,y0)[0]),settled=bool(settled),censored=bool(censored),
        simulated_seconds=float(times[-1]),tail_range_hz=range_,tail_slope_hz_s=slope,
        loss_mw=v['loss'],inertia_mws=v['E'],gain_mw_hz=float(v['gain'].sum()),
        reserve_mw=float(v['rho'].sum()),gain_source=float(F0*v['gain'].sum()),demand_mw=float(demand_mw))
    return record,dict(time_s=times,frequency_hz=freq,response_mw=rtotal) if return_trace else None


def evaluate_event(power,demand_mw,outage_index,commitment=None):
    key=event_key(power,demand_mw,outage_index,commitment);started=time.perf_counter()
    base=dict(event_key=key,model_id=MODEL_ID,outage_unit=IDS[int(outage_index)],outage_index=int(outage_index))
    try:
        coarse,_=integrate_event(power,demand_mw,outage_index,commitment)
        fine,_=integrate_event(power,demand_mw,outage_index,commitment,refined=True)
        delta=abs(coarse['minimum_observed_hz']-fine['minimum_observed_hz'])
        agree=(coarse['minimum_observed_hz']<FMIN)==(fine['minimum_observed_hz']<FMIN)
        numerical=bool(agree and delta<=NUMERIC['nadir_refine_tol_hz'] and coarse['censored']==fine['censored'])
        witnessed=bool(fine['minimum_observed_hz']<FMIN-NUMERIC['boundary_hz'])
        resolved=bool(numerical and fine['settled'] and not fine['censored'])
        if numerical and witnessed:status='UNSAFE_OBSERVED';label=-1
        elif resolved and fine['minimum_observed_hz']>=FMIN+NUMERIC['boundary_hz']:status='SAFE_CONDITIONAL';label=1
        else:status='REVIEW';label=None
        base.update(fine,status=status,y_safe=label,numerical_refinement_ok=numerical,nadir_refinement_abs_hz=delta,
            nadir_crossing_observed=witnessed,resolved_nadir=resolved,
            rocof_crossing=bool(abs(fine['initial_rocof_hz_s'])>CASE['rocof_max_hz_s']+1e-6),
            qss_unresolved=not fine['settled'],qss_crossing=bool(fine['settled'] and abs(F0-fine['endpoint_hz'])>CASE['qss_drop_hz']+1e-6),
            review_reason='' if status!='REVIEW' else ('numerical_refinement' if not numerical else 'unsettled_or_boundary'),
            error='')
    except Exception as exc:
        base.update(status='FAILED_RETAINED',y_safe=None,numerical_refinement_ok=False,nadir_crossing_observed=False,
                    resolved_nadir=False,qss_unresolved=True,error=repr(exc))
    base['calculation_seconds']=time.perf_counter()-started
    return base


def cache_open(path):
    path=Path(path);cache={}
    if path.exists():
        lines=path.read_text(encoding='utf-8').splitlines()
        for i,line in enumerate(lines):
            if not line.strip():continue
            try:row=json.loads(line)
            except json.JSONDecodeError:
                if i==len(lines)-1:
                    backup=path.with_suffix('.truncated_backup_'+str(time.time_ns()))
                    shutil.copyfile(path,backup)
                    path.write_text('\n'.join(lines[:i])+'\n',encoding='utf-8');break
                raise ValueError('Corrupt cache before its last line; preserve file for inspection')
            if row.get('model_id')!=MODEL_ID:raise ValueError('Cache physics/numerics identity differs; never reuse these labels')
            cache[row['event_key']]=row
    return cache

def cache_save(handle,row):
    handle.write(json.dumps(row,allow_nan=False,default=lambda x:x.item() if isinstance(x,np.generic) else str(x))+'\n');handle.flush()




### Source code stage 4

**Algorithm objects:** `class_check`, `select_fit`, `fit_model`, `probability`, `wilson`, `metrics`, `parent_bootstrap_metrics`, `reliability`, `derive_cut`, `_gamma_roundoff`, `audit_frozen_cut`, `group_folds`, `sigmoid_from_oof`, `fit_crosscal`, `train35`.

These definitions implement the original routines; inspect their input validation and return fields before calling them. Defining a function does not itself verify its numerical behavior.

**Verification:** original code-cell text is unchanged and syntax-checked. Printed values and figures become new evidence only after their required inputs have been supplied and execution actually succeeds. Preserve caches/receipts and distinguish a failed calculation from an unsafe physical label.

In [ ]:
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression, SGDClassifier, PassiveAggressiveClassifier
from sklearn.svm import SVC
from scipy.optimize import minimize
from scipy.special import logit
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import confusion_matrix, average_precision_score, roc_auc_score, brier_score_loss
import joblib, warnings


LEARN_CONFIG=dict(seed=20260920,outer_folds=3,repeats=2,inner_folds=3,
    final_roles=['development','calibration','selection','test'],
    calibration_pool_roles=['development','calibration'],
    policies=['cheap','random'],thresholds=[.50,.90,.95,.99],primary_policy='cheap',
    primary_features=CORE_FEATURES,include_threshold_weeks=True,
    recipe='parent-grouped out-of-fold sigmoid; one final estimator, no probability ensemble',
    branch_version=BRANCH_VERSION,minimum_calibration_per_class=10)
EXPORT_CHECK_VERSION='LP35_FROZEN_AFFINE_AUDIT_v3_component_probability'
BUILDERS={
'LR_C1':lambda:LogisticRegression(C=1,solver='lbfgs',max_iter=10000),
'LR_C1e6':lambda:LogisticRegression(C=1e6,solver='lbfgs',max_iter=20000),
'SVM_C01':lambda:SVC(C=.1,kernel='linear'),
'SVM_C1':lambda:SVC(C=1,kernel='linear'),
'SVM_C10':lambda:SVC(C=10,kernel='linear'),
'LDA':lambda:LinearDiscriminantAnalysis(solver='svd'),
'SGD_Log':lambda:SGDClassifier(loss='log_loss',random_state=LEARN_CONFIG['seed'],max_iter=20000,tol=1e-7),
'SGD_Hinge':lambda:SGDClassifier(loss='hinge',random_state=LEARN_CONFIG['seed'],max_iter=20000,tol=1e-7),
'PA':lambda:PassiveAggressiveClassifier(C=1,random_state=LEARN_CONFIG['seed'],max_iter=20000,tol=1e-7)}


def class_check(frame,name):
    counts=frame.y_safe.value_counts()
    if len(counts)!=2 or counts.min()<2:raise ValueError(f'{name}: insufficient usable records of both classes: {counts.to_dict()}')


def select_fit(frame,policy,seed):
    # Selection acts only on candidate fitting parents, never on calibration/test.
    parent=frame[['parent_id','demand_band','screen_cost_keur_h']].drop_duplicates('parent_id')
    chosen=[];rng=np.random.default_rng(seed)
    for _,band in parent.groupby('demand_band'):
        n=max(1,len(band)//2)
        if policy=='cheap':sub=band.sort_values(['screen_cost_keur_h','parent_id']).head(n)
        elif policy=='random':sub=band.iloc[rng.permutation(len(band))[:n]]
        else:raise ValueError(policy)
        chosen.extend(sub.parent_id)
    out=frame[frame.parent_id.isin(chosen)].copy();class_check(out,'Selected fitting records');return out


def fit_model(frame,name):
    class_check(frame,'Base fitting')
    pipe=Pipeline([('scale',StandardScaler()),('model',BUILDERS[name]())])
    start=time.perf_counter()
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter('always');pipe.fit(frame[CORE_FEATURES].to_numpy(),frame.y_safe.to_numpy(int))
    return pipe,time.perf_counter()-start,[str(a.message) for a in w]


def probability(pipe,cal,frame):
    return expit(cal['a']*pipe.decision_function(frame[CORE_FEATURES].to_numpy())+cal['b'])


def wilson(k,n):
    if n==0:return np.nan,np.nan
    z=1.95996398454;h=z*z/n;p=k/n;d=1+h
    centre=(p+h/2)/d;half=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))/d
    return max(0,centre-half),min(1,centre+half)


def metrics(frame,p,pstar):
    unsafe=(frame.y_safe.to_numpy()==-1).astype(int);reject=(np.asarray(p)<pstar).astype(int)
    tn,fp,fn,tp=confusion_matrix(unsafe,reject,labels=[0,1]).ravel();lo,hi=wilson(int(fn),int(fn+tp))
    pp=np.asarray(p);binid=np.minimum((pp*10).astype(int),9);safe=1-unsafe
    ece=sum(float((binid==k).mean())*abs(float(pp[binid==k].mean())-float(safe[binid==k].mean())) for k in range(10) if (binid==k).any())
    return dict(n=len(frame),unsafe_n=int(unsafe.sum()),TP=int(tp),FN=int(fn),FP=int(fp),TN=int(tn),
        recall_unsafe=tp/(tp+fn) if tp+fn else np.nan,FNR=fn/(fn+tp) if fn+tp else np.nan,
        FNR_Wilson_lo=lo,FNR_Wilson_hi=hi,FPR=fp/(fp+tn) if fp+tn else np.nan,
        precision_unsafe=tp/(tp+fp) if tp+fp else 0.,F1_unsafe=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.,
        accuracy=(tp+tn)/len(frame),ECE_10bins=ece,Brier=float(brier_score_loss(1-unsafe,p)),
        AP_unsafe=float(average_precision_score(unsafe,1-np.asarray(p))),
        ROC_AUC_unsafe=float(roc_auc_score(unsafe,1-np.asarray(p))) if len(np.unique(unsafe))==2 else np.nan,
        pstar=pstar)


def parent_bootstrap_metrics(frame, probabilities, pstar=.99, repetitions=400, seed=20260920):
    """Resample entire parent groups; never treat related outages as independent.
    These are conditional sampling intervals, not physical-parameter uncertainty.
    """
    unsafe=(frame.y_safe.to_numpy()==-1);pred=np.asarray(probabilities)<pstar
    codes,unique=pd.factorize(frame.parent_id,sort=True);n=len(unique)
    if n<2:return {'bootstrap_parents':n,'bootstrap_repetitions':0}
    fields={'FN':unsafe&~pred,'TP':unsafe&pred,'FP':~unsafe&pred,'TN':~unsafe&~pred,
            'n':np.ones(len(frame)), 'squared_error':(np.asarray(probabilities)-(~unsafe).astype(int))**2}
    sums={key:np.bincount(codes,weights=np.asarray(value,float),minlength=n) for key,value in fields.items()}
    draw=np.random.default_rng(seed).integers(0,n,size=(repetitions,n))
    vals={key:value[draw].sum(1) for key,value in sums.items()}
    result={'bootstrap_parents':n,'bootstrap_repetitions':repetitions}
    for key,numerator,denominator in [('FNR',vals['FN'],vals['FN']+vals['TP']),('FPR',vals['FP'],vals['FP']+vals['TN']),('Brier',vals['squared_error'],vals['n'])]:
        a=np.divide(numerator,denominator,out=np.full_like(numerator,np.nan),where=denominator>0)
        result[key+'_parent_bootstrap_lo95']=float(np.nanquantile(a,.025));result[key+'_parent_bootstrap_hi95']=float(np.nanquantile(a,.975))
    return result


def reliability(frame,p,candidate):
    edges=np.r_[np.arange(0,1,.1),1.00000000001];idx=np.digitize(p,edges)-1;rows=[]
    for i in range(10):
        sel=idx==i;n=int(sel.sum())
        if n:
            safe=(frame.y_safe.to_numpy()[sel]==1);lo,hi=wilson(int(safe.sum()),n)
            rows.append(dict(candidate=candidate,bin=i,n=n,mean_probability_safe=float(np.asarray(p)[sel].mean()),
                             observed_safe_fraction=float(safe.mean()),low95=lo,high95=hi))
    return rows


def derive_cut(artifact,pstar):
    pipe=artifact['pipeline'];cal=artifact['calibrator'];scale=pipe['scale'];m=pipe['model']
    w=np.ravel(m.coef_).astype(float);bias=float(np.ravel(m.intercept_)[0])
    c=cal['a']*w/scale.scale_;c0=cal['a']*bias+cal['b']-math.log(pstar/(1-pstar))-float(c@scale.mean_)
    # Test the actual twelve-significant-digit serialization, not just unrounded floats.
    fmt=lambda v:float(f'{v:.12g}')
    c=np.array([fmt(v) for v in c]);c0=fmt(c0)
    M=[]
    for outage in range(11):
        mask=np.arange(11)!=outage
        # Inactive candidate: its lost output is exactly zero.
        lower=c0+min(0,c[1])*EUNIT[mask].sum()+min(0,c[2])*GAIN[mask].sum()+min(0,c[3])*PMAX[mask].sum()
        M.append(fmt(max(0,-lower)+1e-5*(1+abs(lower))))
    return dict(k0=c0,kL=float(c[0]),kE=float(c[1]),kG=float(c[2]),kR=float(c[3]),bigM_by_outage=M,pstar=pstar)


def _gamma_roundoff(n):
    eps=np.finfo(np.float64).eps
    if n*eps>=.01:raise ValueError('Invalid floating-point operation-count bound')
    return n*eps/(1.-n*eps)


def audit_frozen_cut(artifact,cut,frame,run,partition):
    """Separate native/primal evaluation, unit conversion and decimal serialization.
    Zero acceptance-decision changes are required. Native SVC dual evaluation
    and its four-weight primal representation can differ by roundoff, amplified
    by the saved Platt slope. That is checked using the EXISTING forward-error
    budget, independently of decimal serialization. The 1e-6 probability ceiling
    is retained for primal-to-export conversion/serialization, where it belongs;
    native-to-export probability error is bounded using sigmoid's derivative
    (at most 1/4) and the independently computed total arithmetic budget.
    No weights, calibrator, thresholds, or GAMS tolerance are altered.
    Cases whose sign is not separated from the arithmetic budget are rejected.
    """
    pipe=artifact['pipeline'];sc=pipe['scale'];model=pipe['model'];cal=artifact['calibrator']
    X=frame[CORE_FEATURES].to_numpy(dtype=float)
    if not len(X) or not np.isfinite(X).all():raise ValueError('Nonfinite/empty export-verification inputs')
    if not np.array_equal(np.asarray(model.classes_),np.array([-1,1])):
        raise ValueError('Expected binary [-1,+1] score orientation; do not reverse an unverified classifier')
    w=np.asarray(model.coef_,dtype=float).reshape(-1)
    bias=float(np.asarray(model.intercept_).reshape(-1)[0])
    a=float(cal['a']);b=float(cal['b']);q=float(cut['pstar']);threshold=math.log(q/(1-q))
    if len(w)!=len(CORE_FEATURES) or not np.isfinite(np.r_[w,bias,a,b]).all() or a<=0:
        raise ValueError('Invalid affine parameters or non-increasing calibrator')
    Z=sc.transform(X)
    native=np.asarray(pipe.decision_function(X),float).reshape(-1)
    primal=Z@w+bias
    # Exactly the same unrounded algebra as derive_cut(), retained unchanged.
    c=a*w/sc.scale_
    intercept=a*bias+b-threshold-float(c@sc.mean_)
    raw64=X@c+intercept
    rounded=np.array([cut['kL'],cut['kE'],cut['kG'],cut['kR']],float)
    exact_text=np.array([float(f'{value:.12g}') for value in c])
    exact_intercept=float(f'{intercept:.12g}')
    if not np.array_equal(rounded,exact_text) or float(cut['k0'])!=exact_intercept:
        raise ValueError(f'{run}: cut is not the declared twelve-digit serialization of the frozen coefficients')
    exported=X@rounded+float(cut['k0'])
    native_margin=a*native+b-threshold
    primal_margin=a*primal+b-threshold
    P_native=expit(a*native+b)
    P_primal=expit(a*primal+b)
    P_export=expit(exported+threshold)
    if not all(np.isfinite(v).all() for v in [native,primal,raw64,exported,native_margin,primal_margin]):
        raise ValueError(f'{run}: nonfinite score/expanded coefficient arithmetic; do not export')
    d=X.shape[1];safe=8.;tiny=1e-10
    # Exact linear-SVC primal/dual identity is susceptible to cancellation.
    # Bound both coefficient contraction and support-vector score evaluation
    # using absolute terms, rather than assuming their outputs are bit-identical.
    score_terms=np.abs(Z)@np.abs(w)+abs(bias)
    if isinstance(model,SVC):
        if model.kernel!='linear':raise ValueError('A nonlinear SVC is not a four-coefficient affine cut')
        sv=np.asarray(model.support_vectors_,float);dual=np.asarray(model.dual_coef_,float).reshape(-1)
        v=np.abs(dual)@np.abs(sv)
        dual_terms=np.abs(Z)@v+abs(bias)
        score_budget=safe*(_gamma_roundoff(len(dual)+d+5)*dual_terms
                           +_gamma_roundoff(d+3)*score_terms)
        n_support=len(dual)
    else:
        score_budget=safe*_gamma_roundoff(d+3)*score_terms
        n_support=0
    evaluation_budget=abs(a)*score_budget+safe*_gamma_roundoff(6)*(
        abs(a)*(np.abs(native)+np.abs(primal))+2*abs(b)+2*abs(threshold))+tiny
    # Bounds for subtract/divide standardization and expanding its affine rule.
    transform_terms=(np.abs(X)+np.abs(sc.mean_))@(abs(a)*np.abs(w)/sc.scale_)
    conversion_budget=safe*_gamma_roundoff(4*d+16)*(
        transform_terms+abs(a*bias)+abs(b)+abs(threshold)+abs(intercept))+tiny
    # Decimal error is evaluated directly from the serialized coefficients.
    decimal_budget=np.abs(X)@np.abs(rounded-c)+abs(float(cut['k0'])-intercept)
    serialization_budget=decimal_budget+safe*_gamma_roundoff(2*d+8)*(
        np.abs(X)@(np.abs(rounded)+np.abs(c))+abs(float(cut['k0']))+abs(intercept))+tiny
    evaluation_error=np.abs(primal_margin-native_margin)
    conversion_error=np.abs(raw64-primal_margin)
    serialization_error=np.abs(exported-raw64)
    total_budget=evaluation_budget+conversion_budget+serialization_budget
    decision_native=native_margin>=0;decision_export=exported>=0
    mismatch=(decision_native!=decision_export)|(decision_export!=(P_native>=q))
    probability_error=np.abs(P_native-P_export)
    # OLD BUG: a fixed 1e-6 ceiling on native-vs-export probabilities falsely
    # rejected your SVM_C10's finite-arithmetic dual/primal difference. The
    # saved audit proves decimal rounding itself was only ~7.23e-11 in logit.
    # Use the pre-existing independently calculated arithmetic allowance here,
    # not a tolerance estimated from the observed discrepancy.
    sigmoid_roundoff = 64*np.finfo(np.float64).eps
    probability_roundoff_budget = 0.25*total_budget + sigmoid_roundoff
    export_probability_error = np.abs(P_primal-P_export)
    export_probability_budget = 0.25*(conversion_budget+serialization_budget)+sigmoid_roundoff
    ambiguous_sign = np.abs(native_margin) <= total_budget
    failed=(evaluation_error>evaluation_budget)|(conversion_error>conversion_budget)|(
        serialization_error>serialization_budget)|mismatch|ambiguous_sign|(
        probability_error>probability_roundoff_budget)|(
        export_probability_error>export_probability_budget)|(export_probability_error>1e-6)
    columns={'run':run,'partition':partition,'native_margin':native_margin,
        'standardized_primal_margin':primal_margin,'raw_float64_margin':raw64,'serialized_margin':exported,
        'native_primal_error':evaluation_error,'native_primal_budget':evaluation_budget,
        'unit_conversion_error':conversion_error,'unit_conversion_budget':conversion_budget,
        'decimal_export_error':serialization_error,'decimal_export_budget':serialization_budget,
        'probability_error':probability_error,'native_probability_budget':probability_roundoff_budget,
        'primal_export_probability_error':export_probability_error,
        'primal_export_probability_budget':export_probability_budget,
        'ambiguous_sign':ambiguous_sign,'decision_mismatch':mismatch,'guard_failed':failed}
    detail=pd.DataFrame(columns)
    for name in ['parent_id','event_key']:
        if name in frame:detail[name]=frame[name].to_numpy()
    auditdir=LEARN/'Export_Audit_RepresentationFix';auditdir.mkdir(exist_ok=True)
    atomic_csv(detail,auditdir/f'{run}_{partition}.csv')
    record=dict(version=EXPORT_CHECK_VERSION,run=run,partition=partition,n=len(frame),
        support_vectors=n_support,decimal_significant_digits=12,
        max_margin_error=float(np.max(np.abs(exported-native_margin))),
        max_native_primal_error=float(evaluation_error.max()),max_native_primal_budget=float(evaluation_budget.max()),
        max_unit_conversion_error=float(conversion_error.max()),max_unit_conversion_budget=float(conversion_budget.max()),
        max_serialization_error=float(serialization_error.max()),max_serialization_budget=float(serialization_budget.max()),
        max_total_roundoff_budget=float(total_budget.max()),max_probability_error=float(probability_error.max()),
        max_native_probability_budget=float(probability_roundoff_budget.max()),
        max_primal_export_probability_error=float(export_probability_error.max()),
        max_primal_export_probability_budget=float(export_probability_budget.max()),
        primal_export_probability_ceiling=1e-6,
        probability_policy='native/primal: independent roundoff envelope; unit/decimal: 1e-6 ceiling plus error bound',
        decision_mismatches=int(mismatch.sum()),failed_rows=int(failed.sum()),
        near_roundoff_boundary_rows=int((np.abs(native_margin)<=total_budget).sum()),
        accepted=bool(not failed.any()))
    atomic_json(auditdir/f'{run}_{partition}_summary.json',record)
    if failed.any():
        raise ValueError(f'{run}/{partition}: export audit did not pass; saved diagnostics in {auditdir}. '
                         f'Decision changes={record["decision_mismatches"]}; failed rows={record["failed_rows"]}. '
                         'Do not rerun labelling or fitting, or remove the check. Return the Export_Audit folder.')
    return record
def group_folds(frame,n_splits,seed):
    """Stratify PARENTS by presence of an unsafe event; all their outages stay together."""
    from sklearn.model_selection import StratifiedKFold
    groups=frame.groupby('parent_id',sort=True).y_safe.min()
    if (groups==-1).sum()<n_splits:raise ValueError('Too few distinct unsafe parent groups for the declared folds.')
    cv=StratifiedKFold(n_splits=n_splits,shuffle=True,random_state=seed)
    for train_ids,test_ids in cv.split(groups.index,(groups.to_numpy()==-1).astype(int)):
        trainset=set(groups.index[train_ids]);testset=set(groups.index[test_ids])
        assert not trainset & testset
        yield frame[frame.parent_id.isin(trainset)].copy(),frame[frame.parent_id.isin(testset)].copy()

def sigmoid_from_oof(scores, y):
    """Increasing smoothed Platt fit: numerical repair, not a new calibration model.

    Keeps the original L-BFGS-B fit when it satisfies explicit stationarity checks.
    A failed numerical solve is retried with cancellation-safe loss/derivatives;
    a deterministic profile-root solve is the final fallback. The labels, smoothed
    targets, parent folds, a >= 0 restriction and ten-per-class rule are unchanged.
    A genuinely noninformative (zero-slope) optimum is still rejected.
    """
    from scipy.optimize import brentq

    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(y)
    if (scores.ndim != 1 or labels.ndim != 1 or len(scores) != len(labels)
            or not len(scores)):
        raise ValueError('OOF scores/labels must be matching nonempty 1-D arrays.')
    if not np.isfinite(scores).all() or not np.isin(labels, [-1, 1]).all():
        raise ValueError('OOF scores must be finite; labels must be exactly -1/+1.')
    labels = labels.astype(int)
    pos = int((labels == 1).sum())
    neg = int((labels == -1).sum())
    if min(pos, neg) < 10:
        raise ValueError(f'OOF sigmoid has {pos} safe and {neg} unsafe; need >=10 each.')
    mu = float(scores.mean())
    sd = max(float(scores.std()), 1e-10)
    z = (scores - mu) / sd
    if not np.isfinite(z).all() or np.ptp(z) == 0:
        raise ValueError('Noninformative constant OOF scores; no constant acceptance rule exported.')
    target = np.where(labels == 1, (pos + 1) / (pos + 2), 1 / (neg + 2))
    initial = np.array([1., logit((pos + 1) / (len(labels) + 2))], dtype=float)
    attempts = []

    def legacy_objective(v):
        eta = v[0] * z + v[1]
        r = expit(eta) - target
        return float(np.mean(np.logaddexp(0, eta) - target * eta)), np.array([r @ z, r.sum()]) / len(labels)

    def stable_objective(v, hessian=False):
        eta = v[0] * z + v[1]
        # Algebraically the same cross-entropy; no subtraction of nearly equal terms.
        losses = (1 - target) * np.logaddexp(0., eta) + target * np.logaddexp(0., -eta)
        # expit(eta) - target, evaluated without cancellation when expit ~= 1.
        r = np.where(eta >= 0, (1 - target) - expit(-eta), expit(eta) - target)
        gradient = np.array([np.mean(r * z), np.mean(r)])
        value = float(np.mean(losses))
        if not hessian:
            return value, gradient
        weight = expit(eta) * expit(-eta)
        H = np.array([[np.mean(weight * z * z), np.mean(weight * z)],
                      [np.mean(weight * z), np.mean(weight)]])
        return value, gradient, H

    intercept0 = float(logit(float(target.mean())))
    base_loss, base_grad = stable_objective([0., intercept0])
    # For this convex problem, a nonnegative derivative at the lower bound means
    # a constant optimum. Do not conceal it with a made-up small positive slope.
    if base_grad[0] >= 0:
        raise ValueError('Noninformative OOF sigmoid: constrained optimum has zero slope; no constant acceptance rule exported.')

    def certificate(v):
        v = np.asarray(v, dtype=float)
        if v.shape != (2,) or not np.isfinite(v).all() or v[0] <= 0:
            return False, dict(reason='nonfinite or nonpositive slope')
        value, gradient, H = stable_objective(v, hessian=True)
        try:
            correction = np.linalg.solve(H, gradient)
            correction_on_scores = float(np.max(np.abs(correction[0] * z + correction[1])))
        except np.linalg.LinAlgError:
            correction_on_scores = float('inf')
        raw_a = float(v[0] / sd)
        raw_b = float(v[1] - v[0] * mu / sd)
        info = dict(mean_cross_entropy=value, gradient_inf=float(np.max(np.abs(gradient))),
                    max_logit_newton_correction=correction_on_scores,
                    constant_model_loss=base_loss, standardized_slope=float(v[0]),
                    standardized_intercept=float(v[1]), score_mean=mu, score_std_used=sd)
        passed = (np.isfinite([raw_a, raw_b]).all() and raw_a > 1e-12
                  and info['gradient_inf'] <= 1e-9
                  and correction_on_scores <= 1e-6
                  and value <= base_loss + 1e-13)
        return bool(passed), info

    def run_lbfgs(fun, x0, name, maxls):
        opt = minimize(fun, x0, jac=True, method='L-BFGS-B',
                       bounds=[(0., None), (None, None)],
                       options={'maxiter': 10000, 'ftol': 1e-14, 'gtol': 1e-10,
                                'maxls': maxls})
        checked, evidence = certificate(opt.x)
        attempts.append(dict(method=name, optimizer_success=bool(opt.success),
                             message=str(opt.message), iterations=int(opt.nit),
                             certificate_passed=checked, **evidence))
        return opt, bool(opt.success and checked)

    opt, accepted = run_lbfgs(legacy_objective, initial, 'original_LBFGSB', 20)
    numerical_method = 'original_LBFGSB'
    v = np.asarray(opt.x, dtype=float)
    if not accepted:
        start = v if np.isfinite(v).all() and v[0] >= 0 else initial
        opt2, accepted = run_lbfgs(stable_objective, start, 'stable_LBFGSB', 100)
        v = np.asarray(opt2.x, dtype=float)
        numerical_method = 'stable_LBFGSB'

    if not accepted:
        # Profile out the intercept B(A). Its derivative is monotone in B.
        # The derivative of the resulting convex 1-D function is monotone in A.
        # Smoothed targets are strictly between 0 and 1, so a finite optimum exists
        # for nonconstant, positively associated scores.
        zmin, zmax = float(z.min()), float(z.max())

        def intercept_for(slope):
            if slope == 0:
                return intercept0
            lo = intercept0 - slope * zmax - 1.
            hi = intercept0 - slope * zmin + 1.
            def balance(bias):
                return float(stable_objective([slope, bias])[1][1])
            return float(brentq(balance, lo, hi, xtol=1e-12, rtol=1e-14, maxiter=500))

        def profiled_gradient(slope):
            bias = intercept_for(float(slope))
            return float(stable_objective([slope, bias])[1][0])

        upper = max(1., float(v[0]) if np.isfinite(v[0]) else 1.)
        for _ in range(80):
            if profiled_gradient(upper) > 0:
                break
            upper *= 2.
        else:
            raise ValueError('OOF sigmoid profile could not bracket a positive-slope optimum; no rule exported.')
        slope = float(brentq(profiled_gradient, 0., upper, xtol=1e-12, rtol=1e-13, maxiter=500))
        v = np.array([slope, intercept_for(slope)])
        accepted, evidence = certificate(v)
        numerical_method = 'stable_profile_root'
        attempts.append(dict(method=numerical_method, optimizer_success=True,
                             message='Two bracketed scalar roots; same convex objective',
                             certificate_passed=accepted, **evidence))
        if not accepted:
            raise ValueError(f'OOF sigmoid numerical certificate failed after deterministic fallback: {evidence}. No rule exported.')

    checked, evidence = certificate(v)
    if not checked:
        raise ValueError(f'OOF sigmoid failed final numerical certificate: {evidence}.')
    a = float(v[0] / sd)
    b = float(v[1] - v[0] * mu / sd)
    raw_eta = a * scores + b
    normalized_eta = v[0] * z + v[1]
    representation_error = float(np.max(np.abs(expit(raw_eta) - expit(normalized_eta))))
    if not np.isfinite(raw_eta).all() or representation_error > 1e-8:
        raise ValueError('OOF sigmoid raw-score conversion is numerically unstable; no rule exported.')
    if numerical_method != 'original_LBFGSB':
        print('Calibration numerical fallback passed:', numerical_method,
              '| gradient =', f'{evidence["gradient_inf"]:.3g}', flush=True)
    return dict(a=a, b=b, smoothing=True, minimum_per_class=10,
                calibration_safe=pos, calibration_unsafe=neg,
                source='grouped out-of-fold scores',
                optimizer_message=attempts[-1]['message'],
                numerical_revision='LP35_PLATT_STABLE_RETRY_v1',
                numerical_method=numerical_method, attempts=attempts,
                optimality_check=evidence,
                raw_probability_representation_error=representation_error)

def fit_crosscal(frame,model,policy,seed):
    """Produce ONE full-pool model and ONE sigmoid. NEVER average probabilities.
    A record's calibration score is generated without its parent in that fold's fit.
    """
    frame=frame.reset_index(drop=True);oof=pd.Series(np.nan,index=frame.event_key);members=[];fit_seconds=0.;warnlog=[]
    for fold,(tr,va) in enumerate(group_folds(frame,3,seed),1):
        fit=select_fit(tr,policy,seed+100+fold)
        model_fold,secs,warns=fit_model(fit,model);fit_seconds+=secs
        val_scores=np.asarray(model_fold.decision_function(va[CORE_FEATURES].to_numpy()),float)
        oof.loc[va.event_key]=val_scores
        members.append(dict(fold=fold,fit_parents=sorted(set(fit.parent_id)),excluded_score_parents=sorted(set(va.parent_id)),
                            fit_events=len(fit),fit_unsafe=int((fit.y_safe==-1).sum()),score_events=len(va),score_unsafe=int((va.y_safe==-1).sum())))
        warnlog.extend(warns)
        assert not set(fit.parent_id)&set(va.parent_id)
    if oof.isna().any():raise ValueError('Incomplete out-of-fold scores')
    t=time.perf_counter();cal=sigmoid_from_oof(oof.loc[frame.event_key].to_numpy(),frame.y_safe.to_numpy());cal_seconds=time.perf_counter()-t
    final_fit=select_fit(frame,policy,seed)
    pipe,secs,warns=fit_model(final_fit,model);fit_seconds+=secs;warnlog.extend(warns)
    oof_table=frame[['event_key','parent_id','role','y_safe']].copy();oof_table['oof_score']=oof.loc[frame.event_key].to_numpy()
    return dict(pipeline=pipe,calibrator=cal,features=CORE_FEATURES,fit_groups=sorted(set(final_fit.parent_id)),
        calibration_groups=sorted(set(frame.parent_id)),calibration_kind='OUT_OF_FOLD; final-model fit groups overlap calibration pool BY DESIGN; no fold-fit overlaps its scoring parents',
        fit_seconds=fit_seconds,calibration_seconds=cal_seconds,warnings=warnlog,
        fit_events=len(final_fit),fit_unsafe=int((final_fit.y_safe==-1).sum())),oof_table,members

def train35(labels):
    import sklearn, scipy, platform
    environment=dict(python=platform.python_version(),numpy=np.__version__,scipy=scipy.__version__,pandas=pd.__version__,sklearn=sklearn.__version__)
    receipt=json.loads((WORK/'Labels_receipt.json').read_text());assert receipt['production_complete']
    inputs=dict(model_id=MODEL_ID,labels_sha256=sha_file(WORK/'LP_SODE_Labels.csv'),config=LEARN_CONFIG)
    training_id=hash_object(inputs)[:16];saved=LEARN/'Learning_receipt.json'
    if saved.exists():
        prior=json.loads(saved.read_text())
        if prior['training_id']!=training_id:raise ValueError('3.5-Hz learning configuration changed; do not overwrite prior fits')
        if prior.get('environment')!=environment:raise ValueError('Fitted-model environment differs. Use the same Anaconda environment; do not load incompatible pickles or overwrite fits.')
        for n,h in prior['artifact_hashes'].items():
            if sha_file(LEARN/n)!=h:raise ValueError('Frozen fitted artifact differs: '+n)
        print('Using frozen 3.5-Hz models/calibrators. No refitting.',flush=True);return prior
    frame=labels.dropna(subset=['y_safe']).copy();frame.y_safe=frame.y_safe.astype(int)
    pool=frame[frame.role.isin(['development','calibration'])].copy()
    select=frame[frame.role=='selection'].copy();test=frame[frame.role=='test'].copy()
    assert not set(pool.parent_id)&set(select.parent_id) and not set(pool.parent_id)&set(test.parent_id) and not set(select.parent_id)&set(test.parent_id)
    for f,n in [(pool,'OOF pool'),(select,'Selection'),(test,'Final test')]:class_check(f,n)
    nunsafe=int((pool.y_safe==-1).sum());print('Cross-fitting calibration pool:',len(pool),'events,',nunsafe,'unsafe. Held selection/test kept separate.',flush=True)
    plan=[]
    for rep in range(LEARN_CONFIG['repeats']):
        for fold,(tr,va) in enumerate(group_folds(pool,3,LEARN_CONFIG['seed']+rep),1):
            plan.append(dict(repeat=rep+1,fold=fold,train_unsafe=int((tr.y_safe==-1).sum()),evaluation_unsafe=int((va.y_safe==-1).sum()),train_parents=len(set(tr.parent_id)),evaluation_parents=len(set(va.parent_id))))
    atomic_csv(pd.DataFrame(plan),LEARN/'CV_Support_Plan.csv')
    artifacts={};hashes={};cvrows=[];mrows=[];times=[];failures=[];bins=[];boots=[];preds=[];members_all=[]
    for policy in LEARN_CONFIG['policies']:
        for model in BUILDERS:
            candidate=f'{policy}__core4__{model}';cp=LEARN/(candidate+'.joblib');complete=LEARN/(candidate+'_complete.json')
            if complete.exists():
                checkpoint=json.loads(complete.read_text())
                if checkpoint['training_id']!=training_id or sha_file(cp)!=checkpoint['artifact_sha256']:raise ValueError('Candidate checkpoint conflict')
                art=joblib.load(cp);candidate_cv=pd.read_csv(LEARN/(candidate+'_CV.csv')).to_dict('records')
                candidate_members=json.loads((LEARN/(candidate+'_groups.json')).read_text())
                print('Reusing candidate checkpoint:',candidate,flush=True)
            else:
                candidate_cv=[];candidate_members=[]
                for rep in range(LEARN_CONFIG['repeats']):
                    for fold,(tr,va) in enumerate(group_folds(pool,3,LEARN_CONFIG['seed']+rep),1):
                        print(candidate,'calibrated outer fold',rep+1,fold,flush=True)
                        # Outer assessment never enters the base fitting or the OOF calibrator.
                        try:
                            aa,_,membership=fit_crosscal(tr,model,policy,LEARN_CONFIG['seed']+1000+100*rep+fold)
                            candidate_members.append(dict(stage='outer',repeat=rep+1,fold=fold,inner=membership,evaluation_parents=sorted(set(va.parent_id))))
                            probs=probability(aa['pipeline'],aa['calibrator'],va)
                            for q in LEARN_CONFIG['thresholds']:
                                candidate_cv.append(dict(candidate=candidate,repeat=rep+1,fold=fold,status='COMPLETE',fit_seconds=aa['fit_seconds'],cal_seconds=aa['calibration_seconds'],calibration_unsafe=aa['calibrator']['calibration_unsafe'],**metrics(va,probs,q)))
                        except (ValueError,RuntimeError) as exc:
                            # A failed cross-validation fold is an outcome, not permission to substitute a constant-safe rule.
                            print('RETAINED CV FAILURE:',candidate,rep+1,fold,repr(exc),flush=True)
                            for q in LEARN_CONFIG['thresholds']:
                                candidate_cv.append(dict(candidate=candidate,repeat=rep+1,fold=fold,status='FAILED_RETAINED',pstar=q,error=repr(exc)))
                art,oof,finalmembers=fit_crosscal(pool,model,policy,LEARN_CONFIG['seed'])
                art.update(model_id=MODEL_ID,training_id=training_id,candidate=candidate,selection_groups=sorted(set(select.parent_id)),test_groups=sorted(set(test.parent_id)))
                candidate_members.append(dict(stage='final',inner=finalmembers))
                joblib.dump(art,cp);atomic_csv(oof,LEARN/(candidate+'_OOF.csv'))
                atomic_csv(pd.DataFrame(candidate_cv),LEARN/(candidate+'_CV.csv'));atomic_json(LEARN/(candidate+'_groups.json'),candidate_members)
                atomic_json(complete,dict(training_id=training_id,artifact_sha256=sha_file(cp)))
            artifacts[candidate]=art;hashes[cp.name]=sha_file(cp);cvrows+=candidate_cv;members_all.append(dict(candidate=candidate,groups=candidate_members))
            ps=probability(art['pipeline'],art['calibrator'],select)
            for q in LEARN_CONFIG['thresholds']:mrows.append(dict(candidate=candidate,partition='selection',**metrics(select,ps,q)))
            times.append(dict(candidate=candidate,fit_seconds=art['fit_seconds'],calibration_seconds=art['calibration_seconds'],fit_parents=len(art['fit_groups']),fit_events=art['fit_events'],fit_unsafe=art['fit_unsafe'],calibration_unsafe=art['calibrator']['calibration_unsafe'],warnings='; '.join(art['warnings'])))
    choice=pd.DataFrame(mrows);eligible=choice[(choice.partition=='selection')&(choice.pstar==.99)&choice.candidate.str.startswith('cheap__')]
    reference=eligible.sort_values(['FNR','FPR','Brier','candidate']).iloc[0].candidate
    atomic_json(LEARN/'Reference_selection_frozen.json',dict(candidate=reference,training_id=training_id,selection_unsafe=int((select.y_safe==-1).sum()),basis='Selection FNR, FPR, Brier, name; frozen BEFORE final test and weekly outcomes. Rare-event uncertainty disclosed.'))
    for candidate,art in artifacts.items():
        p=probability(art['pipeline'],art['calibrator'],test)
        for q in LEARN_CONFIG['thresholds']:mrows.append(dict(candidate=candidate,partition='test',**metrics(test,p,q)))
        bins+=reliability(test,p,candidate);boots.append(dict(candidate=candidate,pstar=.99,**parent_bootstrap_metrics(test,p)))
        pred=test[['parent_id','event_key','y_safe','minimum_observed_hz','censored','resolved_nadir']].copy();pred['candidate']=candidate;pred['p_safe']=p;preds.append(pred)
    failures=pd.DataFrame(cvrows).query("status != 'COMPLETE'").drop_duplicates(['candidate','repeat','fold']).to_dict('records')
    tables={'Metrics':pd.DataFrame(mrows),'Calibrated_grouped_CV':pd.DataFrame(cvrows),'Reliability_bins':pd.DataFrame(bins),'Fit_times':pd.DataFrame(times),
        'Parent_bootstrap_CI':pd.DataFrame(boots),'Failed_folds':pd.DataFrame(failures,columns=['candidate','repeat','fold','status','error']),
        'Partition_counts':frame.groupby(['role','y_safe']).size().reset_index(name='events')}
    for name,table in tables.items():atomic_csv(table,LEARN/(name+'.csv'))
    atomic_csv(pd.concat(preds,ignore_index=True),LEARN/'Test_predictions.csv');atomic_json(LEARN/'CV_Group_Membership.json',members_all)
    info=dict(**inputs,environment=environment,training_id=training_id,reference_candidate=reference,artifact_hashes=hashes,final_candidates=len(artifacts),cv_failed=len(failures),
        cv_expected=len(BUILDERS)*len(LEARN_CONFIG['policies'])*3*2,calibration_unsafe=nunsafe,
        final_test_unsafe=int((test.y_safe==-1).sum()),final_selection_unsafe=int((select.y_safe==-1).sum()),
        test_safe_fraction=float((test.y_safe==1).mean()),
        brier_prevalence_reference=float(brier_score_loss((test.y_safe==1).astype(int),np.full(len(test),float((pool.y_safe==1).mean())))),
        statistical_caveat='50 unsafe events remain 50; cross-fitting uses data efficiently but creates no new independent evidence. No <1% physical-risk claim.',
        source_domain='Same historical screened pool. Random fit selection is not random/Monte-Carlo generation.',
        retrospective_scope='Previously examined parent groups; not claimed a newly unseen prospective confirmation dataset.')
    atomic_json(saved,info)
    print('3.5-Hz training complete. Frozen reference:',reference,flush=True)
    return info


## Fit/resume checkpoints. This never launches GAMS.

### Source code stage 6

This cell uses objects created in preceding cells. Check the immediately preceding source explanation and the declared input files before execution.

**Production workload:** this stage may fit models or process a large event/campaign library. It was not run for packaging. Do not execute it merely to use the released frozen coefficients.

**Verification:** original code-cell text is unchanged and syntax-checked. Printed values and figures become new evidence only after their required inputs have been supplied and execution actually succeeds. Preserve caches/receipts and distinguish a failed calculation from an unsafe physical label.

In [ ]:
labels35=pd.read_csv(WORK/'LP_SODE_Labels.csv',low_memory=False)
learning35=train35(labels35)
display(pd.read_csv(LEARN/'Fit_times.csv'))
print('CV failures retained:',learning35['cv_failed'])

### Source code stage 7

**Algorithm objects:** `export35`.

These definitions implement the original routines; inspect their input validation and return fields before calling them. Defining a function does not itself verify its numerical behavior.

**Verification:** original code-cell text is unchanged and syntax-checked. Printed values and figures become new evidence only after their required inputs have been supplied and execution actually succeeds. Preserve caches/receipts and distinguish a failed calculation from an unsafe physical label.

In [ ]:
def export35(info):
    """New separate identity, coefficients and launcher; never edits the 3-Hz deployment."""
    runs=[dict(run=n,mode=2,pstar=.99,candidate='cheap__core4__'+n) for n in BUILDERS]
    if LEARN_CONFIG['include_threshold_weeks']:
        runs += [dict(run=n,mode=2,pstar=q,candidate=info['reference_candidate']) for n,q in [('REF050',.5),('REF090',.9),('REF095',.95)]]
    labels=pd.read_csv(WORK/'LP_SODE_Labels.csv',low_memory=False).dropna(subset=['y_safe'])
    audits=[];cuts={};coefs=[]
    for r in runs:
        a=joblib.load(LEARN/(r['candidate']+'.joblib')); c=derive_cut(a,r['pstar']);cuts[r['run']]=c
        if a['model_id']!=MODEL_ID or a['training_id']!=info['training_id']:raise ValueError('Wrong fitted artifact identity')
        for part in ['selection','test']:
            audits.append(audit_frozen_cut(a,c,labels[labels.role==part],r['run'],part))
        sc=a['pipeline']['scale'];m=a['pipeline']['model'];ww=np.ravel(m.coef_);intercept=float(np.ravel(m.intercept_)[0]);pl=a['calibrator']
        for j,feat in enumerate(CORE_FEATURES):coefs.append(dict(run=r['run'],candidate=r['candidate'],feature=feat,mean=sc.mean_[j],scale=sc.scale_[j],standardized_weight=ww[j],standardized_intercept=intercept,platt_a=pl['a'],platt_b=pl['b'],deployed_intercept=c['k0'],deployed_coefficient=[c['kL'],c['kE'],c['kG'],c['kR']][j],pstar=r['pstar'],design_nadir_hz=FMIN))
        print('3.5-Hz export checked:',r['run'],flush=True)
    # Parameters have exactly the same meaning as the accepted native model.
    text=[f'$setglobal LP_MODEL_ID {MODEL_ID}',f'scalar lpNadirHz /{FMIN:.12g}/;',
          'set lpRun /'+','.join(r['run'] for r in runs)+'/;',
          'parameter lpMode(lpRun),lpProbability(lpRun),lpK0(lpRun),lpKL(lpRun),lpKE(lpRun),lpKG(lpRun),lpKR(lpRun),lpBigM(lpRun,i);',
          'lpMode(lpRun)=2;lpProbability(lpRun)=0;lpBigM(lpRun,i)=0;']
    for r in runs:
        n=r['run'];c=cuts[n];text.append(f"lpProbability('{n}')={r['pstar']:.12g};")
        for v,k in [('lpK0','k0'),('lpKL','kL'),('lpKE','kE'),('lpKG','kG'),('lpKR','kR')]:text.append(f"{v}('{n}')={c[k]:.12g};")
        for j,i in enumerate(IDS):text.append(f"lpBigM('{n}','{i}')={c['bigM_by_outage'][j]:.12g};")
    cuttext='\n'.join(text)+'\n'
    olddep=json.loads((GAMS/'LaPalma_Deployment.json').read_text())
    if olddep['model_id']!=OLD_MODEL_ID:raise ValueError('Baseline-source model differs from the declared 3-Hz campaign')
    # Require the 56 existing baseline days before any ML solve, not another BC/RESERVE campaign.
    baseline_files={}
    for run in ['BC','RESERVE']:
        for season in CASE['seasons']:
            for day in range(1,8):
                folder=GAMS/'Results_Seasonal'/olddep['campaign']/run/season/f'd{day}'
                for p in [folder/'COMPLETE.txt',folder/f'Results_{run}_{season}_d{day}.xlsx']:
                    if not p.is_file():raise FileNotFoundError(f'Expected reusable baseline artifact missing: {p}')
                    baseline_files[p.relative_to(PROJECT).as_posix()]=sha_file(p)
    srcfiles=[GAMS/'LaPalma_FCUC_35Hz.gms',GAMS/'Run_LaPalma_35Hz.gms',GAMS/'Input_Data_LaPalma_Weekly.gms',GAMS/'cplex.opt',DATA/'LP_Case.json',WORK/'LP35_Protocol.json']
    # No include file is changed; numerical input hashes remain part of the identity.
    srcfiles+=sorted(p for p in (GAMS/'inc').glob('*.inc') if p.name in [Path(n).name for n in CASE['source_hashes']] and not p.name.startswith('LP_SP'))
    src={p.relative_to(PROJECT).as_posix():sha_file(p) for p in srcfiles}
    identity=hash_object(dict(training_id=info['training_id'],source=src,runs=runs,cuttext=cuttext,baseline_files=baseline_files))[:12]
    campaign='LP35_'+identity;depfile=GAMS/'LP35_Deployment.json'
    dep=dict(campaign=campaign,model_id=MODEL_ID,training_id=info['training_id'],seasons=CASE['seasons'],runs=runs,cuts=cuts,
             n_daily_jobs=len(runs)*28,sources=src,baseline_campaign=olddep['campaign'],baseline_model_id=OLD_MODEL_ID,
             baseline_source_files=baseline_files,baseline_deployment_sha256=sha_file(GAMS/'LaPalma_Deployment.json'),
             baseline_runs=['BC','RESERVE'],nadir_min_hz=FMIN,nadir_drop_hz=3.5,currency='EUR',raw_to_eur=1000.,
             gap_target=.0125,reslim_seconds=1200,polish_reslim_seconds=600,threads=4,
             acceptance='Gap is not a gate. Requires integer incumbent, accepted native primal residuals, then independent notebook audit.',
             SP_included=False,source_role='Prospective 3.5-Hz re-fit within retrospective benchmark experiment; original outcomes preserved.',
             calibration='3-fold grouped out-of-fold sigmoid on development+calibration; one full-pool estimator; no probability ensemble')
    jobs=['* Generated LP35 jobs. No SP, BC or RESERVE is solved.',
          '* Failed chains are retained, not silently accepted. Other independent chains continue.',
          '$if not set RETRY_FAILED $setglobal RETRY_FAILED 0']
    planned=[]
    for season in CASE['seasons']:
        for r in runs:
            for day in range(1,8):
                out=f'Results_35Hz/{campaign}/{r["run"]}/{season}/d{day}';folder=GAMS/out;folder.mkdir(parents=True,exist_ok=True)
                book=folder/f'Results_{r["run"]}_{season}_d{day}.xlsx'
                if len(str(book.resolve()))>210:raise ValueError('Workbook full path exceeds safe length; shorten the outer project path, not its contents.')
                tag=f'{r["run"]}_{season}_{day}'
                # Compute readiness before entering the job block. No $goto crosses
                # an open $ifthen block (important for the native GAMS preprocessor).
                jobs += ['$setglobal LP35_READY 1',
                         f'$if exist "{out}/COMPLETE.txt" $setglobal LP35_READY 0']
                if day>1:
                    jobs.append(f'$if not exist "Results_35Hz/{campaign}/{r["run"]}/{season}/d{day-1}/COMPLETE.txt" $setglobal LP35_READY 0')
                jobs += [f'$ifthen.retry_{tag} %RETRY_FAILED%==0',
                         f'$if exist "{out}/FAILED_SOLVE.txt" $setglobal LP35_READY 0',
                         f'$endif.retry_{tag}',
                         f'$ifthen.{tag} %LP35_READY%==1',
                         '$setglobal LP35_EXPORT_ONLY 0',
                         f'$if exist "{out}/results.gdx" $setglobal LP35_EXPORT_ONLY 1',
                         f'$call gams "LaPalma_FCUC_35Hz.gms" --LPCAMPAIGN={campaign} --RUN={r["run"]} --SEASON={season} --DAY={day} --PREV={day-1} --EXPORT_ONLY=%LP35_EXPORT_ONLY% lo=2 o="{out}/LP35.lst" lf="{out}/LP35.log"',
                         f'$if errorlevel 1 $abort LP35 native compile/execution/export failure: {tag}; inspect its daily LP35.lst and LP35.log. Numerical results are retained.',
                         f'$endif.{tag}']
                planned.append(dict(run=r['run'],season=season,day=day,previous_day=day-1,design_nadir_hz=FMIN,search_target_gap=.0125,search_reslim_seconds=1200,workbook=book.relative_to(PROJECT).as_posix()))
    jobtext='\n'.join(jobs)+'\n';inc=GAMS/'inc/LP35_Calibrated.inc';jobfile=GAMS/'inc/LP35_Jobs.inc'
    if depfile.exists():
        old=json.loads(depfile.read_text())
        if old['campaign']!=campaign:raise ValueError('Existing LP35 deployment does not match; no frozen native outputs overwritten.')
        if inc.read_text()!=cuttext or jobfile.read_text()!=jobtext:raise ValueError('Frozen LP35 jobs or coefficients changed.')
        print('Reusing exact 3.5-Hz deployment:',campaign,flush=True);return old
    inc.write_text(cuttext,encoding='ascii');jobfile.write_text(jobtext,encoding='ascii')
    dep['cut_sha256']=sha_file(inc);dep['jobs_sha256']=sha_file(jobfile);atomic_json(depfile,dep)
    snap=GAMS/'Results_35Hz'/campaign/'source_snapshot';snap.mkdir(parents=True,exist_ok=True)
    for p in srcfiles+[inc,jobfile,depfile]:
        target=snap/p.relative_to(PROJECT);target.parent.mkdir(parents=True,exist_ok=True);shutil.copyfile(p,target)
    atomic_csv(pd.DataFrame(planned),WORK/'Planned_ML_Runs.csv');atomic_csv(pd.DataFrame(coefs),LEARN/'LP35_Coefficients.csv');atomic_csv(pd.DataFrame(audits),LEARN/'LP35_Export_checks.csv')
    print('Prepared',len(planned),'ML daily jobs; 56 baseline days reused; NO SP jobs. Campaign:',campaign,flush=True)
    return dep


## Export all new ML rules; import baseline identities, not their decision markers

### Source code stage 9

This cell uses objects created in preceding cells. Check the immediately preceding source explanation and the declared input files before execution.

**Verification:** original code-cell text is unchanged and syntax-checked. Printed values and figures become new evidence only after their required inputs have been supplied and execution actually succeeds. Preserve caches/receipts and distinguish a failed calculation from an unsafe physical label.

In [ ]:
deployment35=export35(learning35)
print('336 new daily ML jobs; 56 original BC/RESERVE days will be read in place. No SP jobs.')

## Output-only coefficient and learning workbooks

### Source code stage 11

This cell uses objects created in preceding cells. Check the immediately preceding source explanation and the declared input files before execution.

**Verification:** original code-cell text is unchanged and syntax-checked. Printed values and figures become new evidence only after their required inputs have been supplied and execution actually succeeds. Preserve caches/receipts and distinguish a failed calculation from an unsafe physical label.

In [ ]:
write_excel(LEARN/'LP35_Learning_Results.xlsx',{n:pd.read_csv(LEARN/(n+'.csv')) for n in ['Metrics','Calibrated_grouped_CV','Fit_times','Reliability_bins','Parent_bootstrap_CI','Failed_folds','Partition_counts']})
write_excel(LEARN/'LP35_Coefficients.xlsx',{'Coefficients':pd.read_csv(LEARN/'LP35_Coefficients.csv'),'Numerical_export_checks':pd.read_csv(LEARN/'LP35_Export_checks.csv')})

### Source code stage 12

**Algorithm objects:** `plot_learning`.

These definitions implement the original routines; inspect their input validation and return fields before calling them. Defining a function does not itself verify its numerical behavior.

**Verification:** original code-cell text is unchanged and syntax-checked. Printed values and figures become new evidence only after their required inputs have been supplied and execution actually succeeds. Preserve caches/receipts and distinguish a failed calculation from an unsafe physical label.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve
def plot_learning(info):
    """Only frozen tables/predictions are plotted. No model fitting or ODE calls."""
    figs=LEARN/'Figures';figs.mkdir(exist_ok=True);register=[]
    table=pd.read_csv(LEARN/'Metrics.csv');pred=pd.read_csv(LEARN/'Test_predictions.csv');bins=pd.read_csv(LEARN/'Reliability_bins.csv')
    palette=['#0072B2','#56B4E9','#E69F00','#D55E00','#CC79A7','#009E73','#735F99','#A97B43','#267B78']
    def save(fig,name,caption):
        fig.tight_layout();fig.savefig(figs/(name+'.pdf'),bbox_inches='tight');fig.savefig(figs/(name+'.png'),dpi=600,bbox_inches='tight');plt.show();plt.close(fig);register.append(dict(figure=name,caption=caption))
    for model in BUILDERS:
        candidate='cheap__core4__'+model
        for q in [.50,.99]:
            r=table[(table.candidate==candidate)&(table.partition=='test')&(table.pstar==q)].iloc[0]
            vals=np.array([[r.TP,r.FN],[r.FP,r.TN]],int);den=vals.sum(1,keepdims=True);pct=np.divide(vals,den,out=np.zeros_like(vals,dtype=float),where=den>0)
            fig,ax=plt.subplots(figsize=(5.2,4.2));im=ax.imshow(pct,cmap='Blues',vmin=0,vmax=1)
            for i in range(2):
                for j in range(2):ax.text(j,i,f'{vals[i,j]:,}\n{100*pct[i,j]:.2f}%',ha='center',va='center',color='white' if pct[i,j]>.55 else '#192333')
            ax.set_xticks([0,1],['Reject / unsafe','Accept / safe']);ax.set_yticks([0,1],['Actually unsafe','Actually safe']);ax.set_title(f'{model}; p*={q:.2f}; held-out original test groups');fig.colorbar(im,ax=ax,label='Fraction within actual-class row');save(fig,f'CM_{model}_{q:.2f}','Raw counts plus row-normalized fractions. FN means an unsafe point accepted by the rule.')
    fig,ax=plt.subplots(figsize=(8,5))
    for k,model in enumerate(BUILDERS):
        d=pred[pred.candidate=='cheap__core4__'+model];pr,re,_=precision_recall_curve(d.y_safe==-1,1-d.p_safe);ax.plot(re,pr,color=palette[k],label=model)
    prevalence=float((d.y_safe==-1).mean());ax.axhline(prevalence,ls='--',color='#555555',label='Test unsafe fraction');ax.set(xlabel='Unsafe recall',ylabel='Unsafe precision',title='All nine learners; common held-out population');ax.legend(fontsize=8,ncol=2);save(fig,'Unsafe_PR','Precision-recall curves use unsafe as the event class; baseline is the test prevalence.')
    fig,ax=plt.subplots(figsize=(7.5,5));ax.plot([0,1],[0,1],':',color='#555555')
    for k,model in enumerate(BUILDERS):
        b=bins[bins.candidate=='cheap__core4__'+model];ax.plot(b.mean_probability_safe,b.observed_safe_fraction,'o-',markersize=3,color=palette[k],label=model)
    ax.set(xlabel='Mean predicted safe probability',ylabel='Observed safe fraction',title='Reliability: populated bins only');ax.legend(ncol=2,fontsize=8);save(fig,'Reliability','Bin counts and Wilson intervals are in the CSV. This is calibration against model-based labels, not physical risk calibration.')
    fig,ax=plt.subplots(figsize=(8.5,4.5))
    for k,model in enumerate(['LR_C1','PA']):
        d=table[(table.candidate=='cheap__core4__'+model)&(table.partition=='test')].sort_values('pstar');ax.plot(d.pstar,d.FNR,'o-',color=palette[k],label=model+' missed unsafe');ax.plot(d.pstar,d.FPR,'s--',color=palette[k],label=model+' rejected safe')
    ax.set(xlabel='Safe-probability floor',ylabel='Class-conditional error fraction',title='Threshold tradeoff before optimization');ax.legend(fontsize=8);save(fig,'Threshold_errors','The probability floor shifts the same calibrated score; it is not a guaranteed physical failure probability.')
    d=pred[pred.candidate=='cheap__core4__LR_C1'];fig,ax=plt.subplots(figsize=(8,4.5))
    score=np.log(np.clip(d.p_safe,1e-12,1-1e-12)/(1-np.clip(d.p_safe,1e-12,1-1e-12)))
    for label,col in [(1,'#0072B2'),(-1,'#D55E00')]:
        mask=d.y_safe==label;ax.scatter(score[mask],d.loc[mask,'minimum_observed_hz'],s=8,alpha=.5,color=col,label='Safe label' if label==1 else 'Unsafe label')
    ax.axhline(FMIN,ls='--',color='#333333');ax.axvline(math.log(99),ls=':',color='#333333');ax.set(xlabel='Calibrated logit',ylabel='Observed model minimum (Hz)',title='LR-C1 score and model outcome; not field ground truth');ax.legend();save(fig,'Score_vs_minimum','Model-based minima; a censored value, where present, is a stopping observation rather than an eventual nadir.')
    atomic_csv(pd.DataFrame(register),figs/'Figure_Register.csv');return register


### Source code stage 13

This cell uses objects created in preceding cells. Check the immediately preceding source explanation and the declared input files before execution.

**Verification:** original code-cell text is unchanged and syntax-checked. Printed values and figures become new evidence only after their required inputs have been supplied and execution actually succeeds. Preserve caches/receipts and distinguish a failed calculation from an unsafe physical label.

In [ ]:
learning_figures35=plot_learning(learning35)

## Native GAMS handoff
Close output workbooks. In **GAMS IDE**, set the existing `GAMS_LaPalma` directory as the working directory and run **Run_LaPalma_35Hz.gms** with F9. No terminal commands. Leave accepted 3-Hz files alone.

The search uses a 1.25% target, 1200 s plus up to 600 s fixed-commitment polishing. A larger time-limit gap does **not** reject a feasible audited integer schedule. No-integer/infeasible cases remain failures; another method/season chain continues. Excel-only failures reuse saved GDX on a launcher restart.

### Source code stage 15

This cell uses objects created in preceding cells. Check the immediately preceding source explanation and the declared input files before execution.

**Verification:** original code-cell text is unchanged and syntax-checked. Printed values and figures become new evidence only after their required inputs have been supplied and execution actually succeeds. Preserve caches/receipts and distinguish a failed calculation from an unsafe physical label.

In [ ]:
print('WAITING FOR GAMS IDE:',deployment35['campaign'])
print('Run_LaPalma_35Hz.gms, then 03_LP35_Process_and_Plot.ipynb')